# Kaggriculture 模仿学习 — Colab 训练
克隆榜一回放，训练因子化策略网络并打包提交。

In [ ]:
# 1) 拉取本库（换成你的仓库地址）
REPO = 'https://github.com/leooooeo/kaggriculture-v1.git'
BRANCH = 'claude/kaggriculture-farming-agent-1mhbgh'
!git clone -b $BRANCH $REPO kagg
%cd kagg

In [ ]:
!pip install -q numpy torch kaggle-environments kaggle

In [ ]:
# 2) Kaggle 凭据（把 kaggle.json 上传到 Colab，或粘贴 token）
import os, json
os.makedirs('/root/.kaggle', exist_ok=True)
# 方式A: 上传 kaggle.json 到当前目录后：
# !cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
# 方式B: 直接写入
# open('/root/.kaggle/kaggle.json','w').write(json.dumps({'username':'XXX','key':'YYY'}))
# os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
# 3) 拉榜一回放到 data/replays/
#    先在排行榜/对局页找到榜一的 EPISODE_ID 列表，逐个 replay 下载。
TOP1 = 'TOP1_TEAM_NAME'   # <- 榜一确切队名
EPISODE_IDS = []          # <- 填入榜一的 episode id 列表
import os; os.makedirs('data/replays', exist_ok=True)
for eid in EPISODE_IDS:
    !kaggle competitions replay $eid -p data/replays
!ls data/replays | wc -l

In [ ]:
# 4) 建训练分片
!python -m il.dataset --replays data/replays --out data/shards --team "$TOP1"

In [ ]:
# 5) 训练（GPU）
!python -m il.train --shards data/shards --out models/policy.pt --epochs 60 --batch 512

In [ ]:
# 6) 评估
!python -m il.evaluate --games 6 --opponent random
!python -m il.evaluate --games 4 --opponent new.py

In [ ]:
# 7) 打包 + 提交
!tar -czf submission.tar.gz main.py il/ models/policy.pt
!kaggle competitions submit kaggriculture -f submission.tar.gz -m 'IL clone of TOP1 v1'